<a href="https://colab.research.google.com/github/rcNibedita/De-Novo-Gen/blob/main/05_TAK1_Docking_4O91_AutoDockVina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5 — TAK1 Docking Evaluation with 4O91

## Purpose

This notebook performs the next stage after Notebook 4:

**Notebook 4 final shortlist → TAK1 docking → pose/score evaluation**

The primary receptor is **TAK1 PDB 4O91**, an experimentally determined TAK1 structure containing the type-II inhibitor NG2/NG25 in the DFG-out conformation.

The workflow is intentionally simple and explainable:

1. Set up a Colab-compatible docking environment.
2. Load the final candidates from Notebook 4.
3. Download and inspect 4O91.
4. Prepare the TAK1 receptor.
5. Extract the crystallographic NG2 ligand.
6. Define the docking box from the experimental ligand position.
7. Redock NG2 as a protocol-validation control.
8. Prepare the Chapter 4 molecules in 3D.
9. Dock them with AutoDock Vina 1.2.7.
10. Save docking scores and poses.
11. Compare docking results with the Chapter 4 prioritization.

### Important environment change

The original notebook tried to install the Python `vina` package with pip. In the current Colab Python 3.13 runtime, Vina 1.2.7 has no CPython 3.13 wheel, so pip falls back to the source distribution.

This revised notebook therefore uses the **official precompiled Linux x86_64 Vina 1.2.7 executable** and keeps Python 3.13 for RDKit/Meeko/Biopython/Py3Dmol.

The notebook also installs and verifies dependencies in separate steps rather than importing RDKit before installation.


In [ ]:
# Mount Google Drive so Notebook 5 and all project outputs use the De-Novo folder.
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# Check the Colab runtime.
import sys
print("Python:", sys.version)
print("Executable:", sys.executable)


Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Executable: /usr/bin/python3


In [ ]:
# Install Python-side packages separately.
!pip -q install -U rdkit
!pip -q install -U meeko
!pip -q install -U gemmi
!pip -q install -U biopython
!pip -q install -U py3Dmol


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.3 MB/s eta 0:00:00


In [ ]:
# Verify the Python-side environment and Meeko CLI tools.
import rdkit
from rdkit import Chem
import meeko
import gemmi
import Bio
import py3Dmol
import shutil

print("RDKit      :", rdkit.__version__)
print("Meeko      :", meeko.__version__)
print("Gemmi      :", gemmi.__version__)
print("Biopython  :", Bio.__version__)
print("Py3Dmol    : imported successfully")

for exe in ["mk_prepare_receptor.py", "mk_prepare_ligand.py", "mk_export.py"]:
    path = shutil.which(exe)
    print(f"{exe:28s}:", path if path else "NOT FOUND")
    if path is None:
        raise RuntimeError(f"{exe} was not found after Meeko installation.")


RDKit      : 2026.03.5
Meeko      : 0.8.0
Gemmi      : 0.7.5
Biopython  : 1.88
Py3Dmol    : imported successfully
mk_prepare_receptor.py      : /usr/local/bin/mk_prepare_receptor.py
mk_prepare_ligand.py        : /usr/local/bin/mk_prepare_ligand.py
mk_export.py                : /usr/local/bin/mk_export.py


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Install AutoDock Vina 1.2.7 as the official precompiled Linux x86_64 executable.
# This avoids the Python-3.13 pip/source-build problem.

from pathlib import Path
import os, stat, subprocess, urllib.request

VINA_VERSION = "1.2.7"
VINA_URL = (
    "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/"
    f"v{VINA_VERSION}/vina_{VINA_VERSION}_linux_x86_64"
)
VINA_BIN = Path("/content/vina_1.2.7_linux_x86_64")

if not VINA_BIN.exists():
    print("Downloading AutoDock Vina binary...")
    urllib.request.urlretrieve(VINA_URL, VINA_BIN)

VINA_BIN.chmod(VINA_BIN.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

result = subprocess.run([str(VINA_BIN), "--version"], capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("AutoDock Vina executable could not be started.")

print(result.stdout.strip())


AutoDock Vina v1.2.7


### Environment checkpoint

Run the setup cells from the top in a **fresh Colab runtime**.

The notebook deliberately does **not** install the Python `vina` package because the current Colab Python 3.13 runtime has no compatible Vina 1.2.7 wheel. Instead, it downloads the precompiled Vina 1.2.7 Linux executable.

At this point the following must work:

- RDKit import
- Meeko import
- Gemmi import
- `mk_prepare_receptor.py`
- `mk_prepare_ligand.py`
- `mk_export.py`
- AutoDock Vina 1.2.7 executable

If any of these checks fail, stop before downloading 4O91.


## 2. Configuration and directories

In [ ]:
from pathlib import Path
import os, re, json, math, subprocess, urllib.request, shutil
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

PROJECT_DIR = Path("/content/drive/MyDrive/De-Novo")
WORKDIR = PROJECT_DIR / "Notebook5_TAK1_Docking"
WORKDIR.mkdir(parents=True, exist_ok=True)

PDB_ID = "4O91"
PDB_URL = f"https://files.rcsb.org/download/{PDB_ID}.pdb"

# Notebook 4 output. Change ONLY this filename if your actual Notebook 4 CSV has a different name.
INPUT_CSV = PROJECT_DIR / "Notebook4_docking_candidates.csv"

EXHAUSTIVENESS = 16
NUM_MODES = 9
ENERGY_RANGE = 4.0
VINA_SEED = 2026
VINA_CPU = min(4, os.cpu_count() or 1)

BOX_PADDING = 6.0

print("Project directory :", PROJECT_DIR)
print("Working directory :", WORKDIR)
print("Notebook 4 input  :", INPUT_CSV)
print("Input exists?     :", INPUT_CSV.exists())
print("Vina binary       :", VINA_BIN)
print("Vina CPU threads   :", VINA_CPU)


Project directory : /content/drive/MyDrive/De-Novo
Working directory : /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking
Notebook 4 input  : /content/drive/MyDrive/De-Novo/Notebook4_docking_candidates.csv
Input exists?     : True
Vina binary       : /content/vina_1.2.7_linux_x86_64
Vina CPU threads   : 2


## 3. Load the Notebook 4 final candidates

The expected input is the final docking-candidate table produced by Notebook 4.

The notebook does not regenerate candidates. It preserves the candidate identity and the Chapter 4 properties so that docking results can later be compared with the chemical-space prioritization.

In [ ]:
# Load the final candidate set from Notebook 4.

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Notebook 4 candidate file was not found:\n{INPUT_CSV}\n\n"
        "Check the filename and make sure the CSV is in /content/drive/MyDrive/De-Novo/."
    )

candidates = pd.read_csv(INPUT_CSV)

required_columns = [
    "Candidate_ID",
    "Generation_Category",
    "SMILES",
    "Priority_Score",
]

missing = [c for c in required_columns if c not in candidates.columns]

if missing:
    raise ValueError(f"Missing required Notebook 4 columns: {missing}")

candidates["Mol"] = candidates["SMILES"].apply(Chem.MolFromSmiles)

invalid = candidates["Mol"].isna()

if invalid.any():
    print("Removing invalid SMILES:", int(invalid.sum()))
    candidates = candidates.loc[~invalid].copy()

candidates = candidates.reset_index(drop=True)

print("Candidates loaded:", len(candidates))
print("Input file:", INPUT_CSV)

display(candidates[required_columns].head(20))


Candidates loaded: 30
Input file: /content/drive/MyDrive/De-Novo/Notebook4_docking_candidates.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Candidate_ID,Generation_Category,SMILES,Priority_Score
0,Dock_001,Baseline_A_Unconditional,C[C@H]1CN(C)C[C@H]1C(=O)NC1CN(C(=O)c2cnn(CCF)c...,0.675459
1,Dock_002,Baseline_A_Unconditional,Cn1ncc(C(=O)N2C[C@H]3C[C@H](NC(=O)c4ncc[nH]4)C...,0.669049
2,Dock_003,Baseline_A_Unconditional,CN(C)C(=O)CN1CC[C@@H]2CCN(C(=O)c3cnn(C)c3N)C[C...,0.667738
3,Dock_004,Baseline_A_Unconditional,COc1cc(C(=O)N2CCN(C(=O)[C@@H]3CCN(C)C3)CC2)nn1C,0.665287
4,Dock_005,Baseline_A_Unconditional,O=C(Cn1cccn1)N[C@@H]1CCN(C(=O)c2cc(Cl)no2)C[C@...,0.664408
5,Dock_006,Baseline_A_Unconditional,CC(C)[C@@H](CNC(=O)c1ncn(C)n1)NC(=O)c1cnn2c1OCCC2,0.664091
6,Dock_007,Baseline_A_Unconditional,Cc1cc(CN[C@H]2C[C@@H](NC(=O)c3nnn(C)n3)C23CCC3...,0.663114
7,Dock_008,Baseline_A_Unconditional,O=C(N[C@@H]1CN(C(=O)[C@H]2CCCO2)C[C@@H]1O)c1cc...,0.659319
8,Dock_009,Baseline_A_Unconditional,Cc1ncn(C)c1C(=O)N1C[C@H](O)[C@H](NC(=O)c2cc(F)...,0.656434
9,Dock_010,Baseline_A_Unconditional,Cn1cc(CC(=O)N[C@]23CCC[C@H]2CN(C(=O)[C@@]2(C)C...,0.656177


## 4. Download TAK1 structure 4O91

The experimental ligand in 4O91 is used as the reference for the binding pocket.

We will use the raw PDB file directly and parse it with simple Biopython code. This avoids the ProDy dependency that caused problems in the previous notebook.

In [ ]:
pdb_file = WORKDIR / f"{PDB_ID}.pdb"

if not pdb_file.exists():
    urllib.request.urlretrieve(PDB_URL, pdb_file)

print("Downloaded:", pdb_file)
print("File size:", pdb_file.stat().st_size, "bytes")

Downloaded: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/4O91.pdb
File size: 238707 bytes


In [ ]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure(PDB_ID, str(pdb_file))

print("Models:", len(list(structure)))
for model in structure:
    print("Model", model.id, "chains:", [c.id for c in model])

Models: 1
Model 0 chains: ['A']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 5. Identify the TAK1 chain and NG2 ligand

The following cell inspects the PDB rather than assuming that every non-protein component is the docking ligand.

For 4O91, we expect the crystallographic inhibitor to be **NG2**. The cell reports the observed residue/ligand names so that the selection is transparent.

In [ ]:
# Inspect hetero residues in the first model.
model = next(structure.get_models())

hetero = []

for chain in model:
    for residue in chain:
        hetflag, resseq, icode = residue.id
        if hetflag.strip():
            hetero.append({
                "chain": chain.id,
                "resname": residue.resname.strip(),
                "resseq": resseq,
                "icode": icode
            })

hetero_df = pd.DataFrame(hetero).drop_duplicates()

display(hetero_df.head(50))

,chain,resname,resseq,icode
0,A,NG2,601,
1,A,HOH,701,
2,A,HOH,702,
3,A,HOH,703,
4,A,HOH,704,
5,A,HOH,705,
6,A,HOH,706,
7,A,HOH,707,
8,A,HOH,708,
9,A,HOH,709,


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Identify protein chains and NG2.
protein_chains = []

for chain in model:
    protein_residues = [
        r for r in chain
        if r.id[0] == " " and r.resname.strip() in {
            "ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS",
            "ILE","LEU","LYS","MET","PHE","PRO","SER","THR","TRP",
            "TYR","VAL"
        }
    ]
    if protein_residues:
        protein_chains.append(chain.id)

print("Protein chains:", protein_chains)

ng2_rows = hetero_df[hetero_df["resname"] == "NG2"]
print("\nNG2 records:")
display(ng2_rows)

if ng2_rows.empty:
    raise RuntimeError(
        "NG2 was not found in the downloaded 4O91 PDB. "
        "Inspect the hetero-residue table before proceeding."
    )

NG2_CHAIN = str(ng2_rows.iloc[0]["chain"])
NG2_RESSEQ = int(ng2_rows.iloc[0]["resseq"])

# Prefer chain A if present, otherwise use the first protein chain.
RECEPTOR_CHAIN = "A" if "A" in protein_chains else protein_chains[0]

print("Selected receptor chain:", RECEPTOR_CHAIN)
print("Selected NG2:", NG2_CHAIN, NG2_RESSEQ)

Protein chains: ['A']

NG2 records:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,chain,resname,resseq,icode
0,A,NG2,601,


Selected receptor chain: A
Selected NG2: A 601


## 6. Write the receptor and crystallographic ligand

The receptor contains protein atoms from the selected TAK1 chain only.

The crystallographic NG2 ligand is written separately. This ligand is removed from the receptor before docking but retained as the reference for defining the docking pocket and redocking validation.

In [ ]:
from Bio.PDB import PDBIO, Select

class ProteinChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id

    def accept_chain(self, chain):
        return chain.id == self.chain_id

    def accept_residue(self, residue):
        # Keep standard polymer residues only.
        # This excludes NG2, waters, ions, and other hetero residues.
        hetflag, resseq, icode = residue.id
        return hetflag.strip() == ""

class LigandSelect(Select):
    def __init__(self, chain_id, resname, resseq):
        self.chain_id = chain_id
        self.resname = resname
        self.resseq = resseq

    def accept_chain(self, chain):
        return chain.id == self.chain_id

    def accept_residue(self, residue):
        hetflag, resseq, icode = residue.id
        return (
            hetflag.strip() != ""
            and residue.resname.strip() == self.resname
            and resseq == self.resseq
        )

receptor_pdb = WORKDIR / "4O91_TAK1_receptor.pdb"
ng2_crystal_pdb = WORKDIR / "4O91_NG2_crystal.pdb"

io = PDBIO()
io.set_structure(structure)
io.save(str(receptor_pdb), ProteinChainSelect(RECEPTOR_CHAIN))

io.set_structure(structure)
io.save(
    str(ng2_crystal_pdb),
    LigandSelect(NG2_CHAIN, "NG2", NG2_RESSEQ)
)

print("Protein-only receptor written:", receptor_pdb)
print("NG2 crystal ligand written:", ng2_crystal_pdb)


Protein-only receptor written: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/4O91_TAK1_receptor.pdb
NG2 crystal ligand written: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/4O91_NG2_crystal.pdb


## 7. Define the docking box from the experimental NG2 position

Rather than hard-coding arbitrary coordinates, the box is calculated from the heavy-atom coordinates of the crystallographic NG2 ligand.

The box center is the ligand centroid.

The box dimensions are:

**ligand span + 2 × padding**

This gives the generated molecules enough room to explore the experimentally identified pocket while keeping the search localized.

In [ ]:
# Read NG2 coordinates and calculate the experimental-pocket box.
coords = []

with open(ng2_crystal_pdb) as f:
    for line in f:
        if line.startswith(("ATOM", "HETATM")):
            element = line[76:78].strip()
            if element.upper() == "H":
                continue
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            coords.append([x, y, z])

coords = np.asarray(coords)

if len(coords) == 0:
    raise RuntimeError("No NG2 heavy-atom coordinates were found.")

mins = coords.min(axis=0)
maxs = coords.max(axis=0)
center = (mins + maxs) / 2.0
size = (maxs - mins) + 2 * BOX_PADDING

box = {
    "center_x": float(center[0]),
    "center_y": float(center[1]),
    "center_z": float(center[2]),
    "size_x": float(size[0]),
    "size_y": float(size[1]),
    "size_z": float(size[2]),
}

print("Docking box:")
for k, v in box.items():
    print(f"{k}: {v:.3f}")

Docking box:
center_x: -5.774
center_y: -49.551
center_z: -16.372
size_x: 31.031
size_y: 18.926
size_z: 20.844


## 8. Prepare the TAK1 receptor with Meeko

Meeko converts the receptor to the PDBQT format required by AutoDock Vina.

We use the prepared protein-only receptor. The crystallographic NG2 ligand is not included in this receptor file.

In [ ]:
receptor_prefix = WORKDIR / "4O91_TAK1"

cmd = [
    "mk_prepare_receptor.py",
    "--read_pdb", str(receptor_pdb),
    "-o", str(receptor_prefix),
    "-p",
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Meeko receptor preparation failed.")

receptor_pdbqt = WORKDIR / "4O91_TAK1.pdbqt"

if not receptor_pdbqt.exists():
    raise FileNotFoundError(f"Receptor PDBQT not found: {receptor_pdbqt}")

print("Prepared receptor:", receptor_pdbqt)

Running:
mk_prepare_receptor.py --read_pdb /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/4O91_TAK1_receptor.pdb -o /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/4O91_TAK1 -p


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Prepared receptor: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/4O91_TAK1.pdbqt


## 9. Prepare NG2 for redocking

The crystal ligand is first used to obtain the correct ligand chemistry from the RCSB Chemical Component Dictionary.

The resulting ligand is then converted to PDBQT.

The purpose is not to redesign NG2. It is to create a docking-compatible representation of the known ligand.

In [ ]:
ng2_ideal_sdf = WORKDIR / "NG2_ideal.sdf"
ng2_ideal_url = "https://files.rcsb.org/ligands/download/NG2_ideal.sdf"

if not ng2_ideal_sdf.exists():
    urllib.request.urlretrieve(ng2_ideal_url, ng2_ideal_sdf)

supplier = Chem.SDMolSupplier(str(ng2_ideal_sdf), removeHs=False)
ng2_template = supplier[0]

if ng2_template is None:
    raise RuntimeError("Could not read NG2 ideal structure.")

print("NG2 template atoms:", ng2_template.GetNumAtoms())
print("Formula:", rdMolDescriptors.CalcMolFormula(ng2_template))

[RDKit] WARNING:[07:39:18] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


NG2 template atoms: 69
Formula: C29H30F3N5O2


In [ ]:
# Read crystallographic NG2 coordinates.
ng2_crystal_mol = Chem.MolFromPDBFile(
    str(ng2_crystal_pdb),
    removeHs=True,
    sanitize=False
)

if ng2_crystal_mol is None:
    raise RuntimeError("Could not parse crystallographic NG2 coordinates.")

# Transfer bond orders from the ideal CCD template.
try:
    ng2_redock = AllChem.AssignBondOrdersFromTemplate(
        ng2_template,
        ng2_crystal_mol
    )
    Chem.SanitizeMol(ng2_redock)
    print("Bond orders transferred from CCD template.")
except Exception as e:
    print("Bond-order transfer failed:", repr(e))
    print("Using the ideal NG2 structure as a fallback.")
    ng2_redock = Chem.Mol(ng2_template)

# Add explicit hydrogens.
ng2_redock = Chem.AddHs(ng2_redock, addCoords=True)

# Ensure a 3D conformer exists.
if ng2_redock.GetNumConformers() == 0:
    status = AllChem.EmbedMolecule(ng2_redock, randomSeed=2026)
    if status != 0:
        raise RuntimeError("Could not generate NG2 3D coordinates.")
    AllChem.UFFOptimizeMolecule(ng2_redock)

ng2_sdf = WORKDIR / "NG2_redocking_input.sdf"

writer = Chem.SDWriter(str(ng2_sdf))
writer.write(ng2_redock)
writer.close()

print("NG2 SDF:", ng2_sdf)

Bond-order transfer failed: ValueError('No matching found')
Using the ideal NG2 structure as a fallback.
NG2 SDF: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/NG2_redocking_input.sdf


In [ ]:
ng2_pdbqt = WORKDIR / "NG2_redocking_input.pdbqt"

cmd = [
    "mk_prepare_ligand.py",
    "-i", str(ng2_sdf),
    "-o", str(ng2_pdbqt)
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Meeko NG2 ligand preparation failed.")

print("Prepared NG2:", ng2_pdbqt)

Prepared NG2: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/NG2_redocking_input.pdbqt


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 10. Redock NG2 with AutoDock Vina

This is the protocol-validation experiment.

The notebook uses the **AutoDock Vina 1.2.7 command-line executable**, not the Python `vina` package. This avoids the Python 3.13 wheel problem while using the Vina 1.2.7 docking engine.

If the known ligand cannot produce a plausible pose in its experimentally observed pocket, do not immediately trust the generated-molecule docking results.


In [ ]:
def parse_vina_affinities(log_text):
    # Parse the Vina result table:
    # mode | affinity | rmsd_lb | rmsd_ub

    pattern = re.compile(
        r"^\s*(\d+)\s+(-?\d+(?:\.\d+)?)\s+"
        r"(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)\s*$",
        re.MULTILINE
    )

    rows = []

    for mode, affinity, rmsd_lb, rmsd_ub in pattern.findall(log_text):
        rows.append({
            "Mode": int(mode),
            "Affinity_kcal_mol": float(affinity),
            "RMSD_LB": float(rmsd_lb),
            "RMSD_UB": float(rmsd_ub),
        })

    return pd.DataFrame(rows)


def run_vina(receptor, ligand, output_pdbqt, log_file, seed=VINA_SEED):

    cmd = [
        str(VINA_BIN),

        "--receptor", str(receptor),
        "--ligand", str(ligand),

        "--center_x", str(box["center_x"]),
        "--center_y", str(box["center_y"]),
        "--center_z", str(box["center_z"]),

        "--size_x", str(box["size_x"]),
        "--size_y", str(box["size_y"]),
        "--size_z", str(box["size_z"]),

        "--exhaustiveness", str(EXHAUSTIVENESS),
        "--num_modes", str(NUM_MODES),
        "--energy_range", str(ENERGY_RANGE),

        "--cpu", str(VINA_CPU),
        "--seed", str(seed),

        "--out", str(output_pdbqt),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    # Save Vina's stdout/stderr ourselves because Vina 1.2.7
    # does not support the --log command-line option.
    log_text = (
        "===== AutoDock Vina STDOUT =====\n"
        + result.stdout
        + "\n\n===== AutoDock Vina STDERR =====\n"
        + result.stderr
    )

    log_file.write_text(log_text)

    if result.returncode != 0:

        print("STDOUT:")
        print(result.stdout)

        print("STDERR:")
        print(result.stderr)

        raise RuntimeError("AutoDock Vina docking failed.")

    if not output_pdbqt.exists():
        raise FileNotFoundError(
            f"Vina output was not created: {output_pdbqt}"
        )

    energies = parse_vina_affinities(result.stdout)

    if energies.empty:
        print("Vina output:")
        print(result.stdout)

        raise RuntimeError(
            "Could not parse Vina affinity results from Vina output."
        )

    return energies, result.stdout


print("Vina CLI wrapper ready.")

Vina CLI wrapper ready.


## 11. Redocking validation — visual check

The safest first validation is to compare the redocked NG2 pose with the experimental NG2 coordinates.

Because reliable RMSD requires correct atom-to-atom correspondence, this notebook does not manufacture an RMSD from uncertain atom mapping.

Instead, the redocked pose is exported so it can be inspected directly in a molecular viewer. A later interaction-analysis notebook can perform a rigorous mapped RMSD calculation if required.

In [ ]:
# Redock the crystallographic NG2 ligand as a protocol-validation control.
#
# IMPORTANT:
# This cell must be run before the NG2 PDBQT -> SDF export cell.

ng2_docked_pdbqt = WORKDIR / "NG2_redocked.pdbqt"
ng2_log = WORKDIR / "NG2_redocked.log"

print("Running NG2 redocking...")

ng2_energies, ng2_vina_stdout = run_vina(
    receptor_pdbqt,
    ng2_pdbqt,
    ng2_docked_pdbqt,
    ng2_log,
)

if not ng2_docked_pdbqt.exists():
    raise FileNotFoundError(
        f"Vina completed but the expected output was not found: "
        f"{ng2_docked_pdbqt}"
    )

display(ng2_energies)

print("Redocked NG2 PDBQT:", ng2_docked_pdbqt)
print("Vina log:", ng2_log)
print(
    "Best Vina affinity:",
    float(ng2_energies.iloc[0]["Affinity_kcal_mol"]),
    "kcal/mol"
)


Running NG2 redocking...


,Mode,Affinity_kcal_mol,RMSD_LB,RMSD_UB
0,1,-12.360,0.000,0.00
1,2,-10.650,3.829,11.77
2,3,-9.635,6.775,13.31
3,4,-9.616,7.201,13.32
4,5,-9.526,3.418,11.65
5,6,-9.473,3.512,13.39
6,7,-9.450,3.692,11.54
7,8,-9.389,3.715,11.21
8,9,-9.322,6.475,13.41


Redocked NG2 PDBQT: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/NG2_redocked.pdbqt
Vina log: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/NG2_redocked.log
Best Vina affinity: -12.36 kcal/mol


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Convert the redocked PDBQT to SDF for easier visualization.

if not Path(ng2_docked_pdbqt).exists():
    raise FileNotFoundError(
        f"Redocked NG2 PDBQT was not found: {ng2_docked_pdbqt}"
    )

ng2_redocked_sdf = WORKDIR / "NG2_redocked.sdf"

cmd = [
    "mk_export.py",
    str(ng2_docked_pdbqt),
    "-s", str(ng2_redocked_sdf)
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Could not export redocked NG2.")

if not ng2_redocked_sdf.exists():
    raise FileNotFoundError(
        f"Meeko did not create the expected SDF: {ng2_redocked_sdf}"
    )

print("Redocked NG2 SDF:", ng2_redocked_sdf)
print(
    "Best Vina affinity:",
    float(ng2_energies.iloc[0]["Affinity_kcal_mol"]),
    "kcal/mol"
)


Redocked NG2 SDF: /content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/NG2_redocked.sdf
Best Vina affinity: -12.36 kcal/mol


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Redocking decision

Before moving on, inspect the redocked NG2 pose in a molecular viewer.

We want to see whether the predicted ligand remains in the same binding pocket and adopts a chemically plausible orientation relative to the experimental ligand.

If the redocking is clearly wrong, stop here and investigate the receptor preparation or docking box before interpreting the generated molecules.

## 12. Prepare the Chapter 4 generated molecules

Each candidate is converted from SMILES to a 3D structure.

Process:

**SMILES → add H → ETKDG 3D embedding → light UFF optimization → SDF → PDBQT**

This is a practical ligand-preparation step. It is not molecular dynamics and does not attempt to calculate a physical binding free energy.

In [ ]:
ligand_sdf_dir = WORKDIR / "ligands_sdf"
ligand_pdbqt_dir = WORKDIR / "ligands_pdbqt"

ligand_sdf_dir.mkdir(exist_ok=True)
ligand_pdbqt_dir.mkdir(exist_ok=True)

prep_records = []

for _, row in candidates.iterrows():
    cid = str(row["Candidate_ID"])
    mol = Chem.MolFromSmiles(str(row["SMILES"]))

    if mol is None:
        prep_records.append({
            "Candidate_ID": cid,
            "Preparation_Status": "invalid_smiles"
        })
        continue

    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 2026

    status = AllChem.EmbedMolecule(mol, params)

    if status != 0:
        prep_records.append({
            "Candidate_ID": cid,
            "Preparation_Status": "3d_embedding_failed"
        })
        continue

    try:
        AllChem.UFFOptimizeMolecule(mol, maxIters=200)
    except Exception:
        # UFF failure does not necessarily mean the structure is unusable.
        # We retain the embedded conformer and continue.
        pass

    sdf_path = ligand_sdf_dir / f"{cid}.sdf"
    pdbqt_path = ligand_pdbqt_dir / f"{cid}.pdbqt"

    writer = Chem.SDWriter(str(sdf_path))
    writer.write(mol)
    writer.close()

    cmd = [
        "mk_prepare_ligand.py",
        "-i", str(sdf_path),
        "-o", str(pdbqt_path)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0 or not pdbqt_path.exists():
        prep_records.append({
            "Candidate_ID": cid,
            "Preparation_Status": "meeko_failed",
            "Message": result.stderr[-500:]
        })
    else:
        prep_records.append({
            "Candidate_ID": cid,
            "Preparation_Status": "ok",
            "SDF": str(sdf_path),
            "PDBQT": str(pdbqt_path)
        })

prep_df = pd.DataFrame(prep_records)

display(prep_df["Preparation_Status"].value_counts())
display(prep_df.head())

prep_df.to_csv(
    WORKDIR / "Notebook5_ligand_preparation.csv",
    index=False
)

,count
Preparation_Status,
ok,29
3d_embedding_failed,1


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Candidate_ID,Preparation_Status,SDF,PDBQT
0,Dock_001,ok,/content/drive/MyDrive/De-Novo/Notebook5_TAK1_...,/content/drive/MyDrive/De-Novo/Notebook5_TAK1_...
1,Dock_002,3d_embedding_failed,NaN,NaN
2,Dock_003,ok,/content/drive/MyDrive/De-Novo/Notebook5_TAK1_...,/content/drive/MyDrive/De-Novo/Notebook5_TAK1_...
3,Dock_004,ok,/content/drive/MyDrive/De-Novo/Notebook5_TAK1_...,/content/drive/MyDrive/De-Novo/Notebook5_TAK1_...
4,Dock_005,ok,/content/drive/MyDrive/De-Novo/Notebook5_TAK1_...,/content/drive/MyDrive/De-Novo/Notebook5_TAK1_...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 13. Dock the generated molecules

All candidates use:

- the same TAK1 receptor;
- the same binding box;
- the same Vina scoring function;
- the same exhaustiveness;
- the same number of output poses.

This makes the resulting scores comparable **within this experiment**.

In [ ]:
docking_dir = WORKDIR / "docking_results"
docking_dir.mkdir(exist_ok=True)

docking_records = []

for _, row in candidates.iterrows():

    cid = str(row["Candidate_ID"])
    ligand_file = ligand_pdbqt_dir / f"{cid}.pdbqt"

    base_record = row.drop(labels=["Mol"]).to_dict()

    if not ligand_file.exists():
        base_record["Docking_Status"] = "ligand_preparation_failed"
        docking_records.append(base_record)
        continue

    output_pose = docking_dir / f"{cid}_docked.pdbqt"
    log_file = docking_dir / f"{cid}_docked.log"

    try:
        energies, _ = run_vina(
            receptor_pdbqt,
            ligand_file,
            output_pose,
            log_file,
        )

        base_record["Docking_Status"] = "ok"
        base_record["Best_Vina_Affinity_kcal_mol"] = float(
            energies.iloc[0]["Affinity_kcal_mol"]
        )
        base_record["N_Poses"] = int(len(energies))
        base_record["Pose_File"] = str(output_pose)
        base_record["Log_File"] = str(log_file)

    except Exception as e:
        base_record["Docking_Status"] = "docking_failed"
        base_record["Docking_Error"] = repr(e)

    docking_records.append(base_record)

docking_results = pd.DataFrame(docking_records)

print("Docking status:")
display(docking_results["Docking_Status"].value_counts())


Docking status:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
Docking_Status,
ok,29
ligand_preparation_failed,1


## 14. Rank docking results

A more negative Vina affinity is a more favorable **predicted docking score** within the same protocol.

This table is only a docking ranking. It does **not** replace the Chapter 4 chemical-space ranking, and a Vina score should not be interpreted as an experimentally validated binding free energy.


In [ ]:
successful = docking_results[
    docking_results["Docking_Status"] == "ok"
].copy()

if successful.empty:
    raise RuntimeError("No successful docking results were produced.")

successful = successful.sort_values(
    "Best_Vina_Affinity_kcal_mol",
    ascending=True
).reset_index(drop=True)

display(successful[[
    "Candidate_ID",
    "Generation_Category",
    "Priority_Score",
    "Best_Vina_Affinity_kcal_mol"
]].head(20))

,Candidate_ID,Generation_Category,Priority_Score,Best_Vina_Affinity_kcal_mol
0,Dock_014,Experiment_1_Seeded,0.777239,-9.985
1,Dock_013,Experiment_1_Seeded,0.778114,-9.365
2,Dock_011,Experiment_1_Seeded,0.825199,-8.375
3,Dock_018,Experiment_1_Seeded,0.707060,-8.363
4,Dock_016,Experiment_1_Seeded,0.745903,-8.356
5,Dock_025,Experiment_2_Decorated,0.839907,-8.326
6,Dock_019,Experiment_1_Seeded,0.704952,-8.320
7,Dock_006,Baseline_A_Unconditional,0.664091,-8.281
8,Dock_005,Baseline_A_Unconditional,0.664408,-8.153
9,Dock_028,Experiment_2_Decorated,0.829264,-7.908


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 15. Compare Chapter 4 priority with docking score

This comparison is important because the two stages answer different questions.

### Chapter 4

Prioritizes:

- chemical quality;
- novelty;
- relationship to known TAK1 chemical space;
- drug-like properties;
- diversity.

### Chapter 5

Evaluates:

- predicted TAK1 binding poses;
- docking score;
- occupancy of the experimentally defined pocket.

Therefore, a molecule should not automatically become the final winner merely because it has the most negative docking score.

In [ ]:
comparison_cols = [
    "Candidate_ID",
    "Generation_Category",
    "SMILES",
    "Priority_Score",
    "Best_Vina_Affinity_kcal_mol"
]

extra_cols = [
    c for c in [
        "Novel_vs_Known_TAK1",
        "Max_TAK1_Tanimoto",
        "QED",
        "MW",
        "LogP"
    ] if c in successful.columns
]

display(
    successful[
        comparison_cols[:4] + extra_cols + ["Best_Vina_Affinity_kcal_mol"]
    ]
)

,Candidate_ID,Generation_Category,SMILES,Priority_Score,Novel_vs_Known_TAK1,Max_TAK1_Tanimoto,QED,MW,LogP,Best_Vina_Affinity_kcal_mol
0,Dock_014,Experiment_1_Seeded,O=C(Nc1cccc(-c2nnc[nH]2)n1)Nc1nccc2ccccc12,0.777239,True,0.606557,0.534454,331.339,3.05890,-9.985
1,Dock_013,Experiment_1_Seeded,O=C(Nc1ccc(C(=O)N2CCNCC2)cc1C1CCCCC1)c1csc2c(=...,0.778114,True,0.840580,0.547923,465.579,3.33000,-9.365
2,Dock_011,Experiment_1_Seeded,c1cc2[nH]ncc2cc1-c1cnc2ccc(N3C[C@@H]4C[C@H]3CO...,0.825199,True,0.703125,0.609504,332.367,2.25010,-8.375
3,Dock_018,Experiment_1_Seeded,O=C(N=c1[nH]c2ccccc2[nH]1)c1ccccc1,0.707060,True,0.380000,0.669526,237.262,2.23710,-8.363
4,Dock_016,Experiment_1_Seeded,O=c1[nH]nc2ccc3ccc(-c4ccc[nH]4)cc3n12,0.745903,True,0.627451,0.543415,250.261,2.17090,-8.356
5,Dock_025,Experiment_2_Decorated,CNC(=O)C=Cc1c(C)[nH]c2ncnc(OC3CCC(F)(F)CC3)c12,0.839907,True,0.546875,0.831019,350.369,2.98222,-8.326
6,Dock_019,Experiment_1_Seeded,c1c[nH]c(-c2ncc(-c3c[nH]c4ccccc34)o2)c1,0.704952,True,0.555556,0.567064,249.273,3.81800,-8.320
7,Dock_006,Baseline_A_Unconditional,CC(C)[C@@H](CNC(=O)c1ncn(C)n1)NC(=O)c1cnn2c1OCCC2,0.664091,True,0.231579,0.746878,361.406,-0.02150,-8.281
8,Dock_005,Baseline_A_Unconditional,O=C(Cn1cccn1)N[C@@H]1CCN(C(=O)c2cc(Cl)no2)C[C@...,0.664408,True,0.191919,0.794193,353.766,-0.08370,-8.153
9,Dock_028,Experiment_2_Decorated,Cc1[nH]c2ncnc(OC3CCC(F)(F)CC3)c2c1C#CC(N)=O,0.829264,True,0.515625,0.822125,334.326,2.05982,-7.908


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 16. Export all Chapter 5 results

The main outputs are:

- `Notebook5_docking_results.csv` — all candidates and docking status
- `Notebook5_successful_docking_results.csv` — successful docking runs
- `Notebook5_ligand_preparation.csv` — ligand preparation status
- `Notebook5_redocking_summary.csv` — NG2 validation result
- `NG2_redocked.pdbqt` and `NG2_redocked.sdf` — redocking poses
- `docking_results/*.pdbqt` — generated-molecule docking poses
- `docking_results/*.log` — Vina logs for reproducibility

These files provide the input for the subsequent **pose and interaction analysis**.


In [ ]:
all_results_file = WORKDIR / "Notebook5_docking_results.csv"
successful_file = WORKDIR / "Notebook5_successful_docking_results.csv"
redock_file = WORKDIR / "Notebook5_redocking_summary.csv"

docking_results.to_csv(all_results_file, index=False)
successful.to_csv(successful_file, index=False)

redock_summary = pd.DataFrame([{
    "PDB_ID": PDB_ID,
    "Reference_Ligand": "NG2",
    "Best_Vina_Affinity_kcal_mol": float(
        ng2_energies.iloc[0]["Affinity_kcal_mol"]
    ),
    "Exhaustiveness": EXHAUSTIVENESS,
    "Num_Modes": NUM_MODES,
    "Seed": VINA_SEED,
    "Vina_Version": VINA_VERSION,
}])

redock_summary.to_csv(redock_file, index=False)

print("Saved:")
print(all_results_file)
print(successful_file)
print(WORKDIR / "Notebook5_ligand_preparation.csv")
print(redock_file)
print(WORKDIR / "NG2_redocked.pdbqt")
print(WORKDIR / "NG2_redocked.sdf")
print(WORKDIR / "docking_results")


Saved:
/content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/Notebook5_docking_results.csv
/content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/Notebook5_successful_docking_results.csv
/content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/Notebook5_ligand_preparation.csv
/content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/Notebook5_redocking_summary.csv
/content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/NG2_redocked.pdbqt
/content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/NG2_redocked.sdf
/content/drive/MyDrive/De-Novo/Notebook5_TAK1_Docking/docking_results


# Notebook 5 — What has been accomplished?

The complete computational sequence is now:

**Notebook 3**
→ generate molecules

**Notebook 4**
→ validate, filter, assess novelty/TAK1 chemical-space relationship, prioritize, and diversity-select

**Notebook 5**
→ dock the final candidates into the experimentally observed TAK1 DFG-out pocket represented by 4O91

The next logical analysis is **not another generic filtering step**. It is detailed **pose and interaction analysis**:

- Does the ligand occupy the intended pocket?
- Does it make plausible hinge interactions?
- Does it interact sensibly with the DFG region?
- Are there steric clashes?
- Are the strongest docking candidates also chemically convincing?

The alternative TAK1 structure **4L53 (DFG-in)** can then be used later as a cross-conformation validation step for the strongest candidates.